# Mamba SOH training — Kaggle GPU (v1.4, window=30)

Train standard production model `MambaSOHPredictor` (window=30, GH-54: +cycle_count +soc_percent, 6 features).

**Trước khi chạy:**
1. Settings → Accelerator → **GPU P100/T4**
2. + Add Data → dataset NASA chứa `cleaned_dataset/metadata.csv` + `cleaned_dataset/data/*.csv`
3. (repo private) Add-ons → Secrets → tạo `GITHUB_TOKEN` = GitHub PAT
4. **Push code lên GitHub trước** (branch `dev` đã merge GH-54) — Kaggle clone từ remote, không thấy local uncommitted changes

> Model long-sequence (v2.2, L=4096) đã train thật riêng — không nằm trong notebook này. Notebook này chỉ train model production window=30.

## 1 — GPU check

In [ ]:
!nvidia-smi
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: bật GPU ở Settings -> Accelerator -> GPU P100/T4')

## 2 — Clone branch (push code lên GitHub trước khi chạy cell này)

In [ ]:
import subprocess
BRANCH  = 'dev'   # đổi thành branch của bạn nếu chưa merge (vd feat/GH-54-...) — PHẢI push lên GitHub trước khi chạy cell này
REPO    = '/kaggle/working/ai-module'
URL_PUB = 'https://github.com/GSU26SE55/ai-module.git'
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    url = f'https://{token}@github.com/GSU26SE55/ai-module.git'
except Exception as e:
    print('No GITHUB_TOKEN secret -> thử public clone:', e)
    url = URL_PUB
subprocess.run(['rm', '-rf', REPO])
subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', url, REPO], check=True)
subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', URL_PUB])  # xoá token khỏi remote
print('Branch:', subprocess.check_output(['git','-C',REPO,'branch','--show-current']).decode().strip())
print('Commit:', subprocess.check_output(['git','-C',REPO,'log','-1','--oneline']).decode().strip())

## 3 — Dependencies (torch đã có sẵn trên Kaggle)

In [ ]:
%pip install -q scipy scikit-learn joblib pandas
import scipy, sklearn; print('scipy', scipy.__version__, '| sklearn', sklearn.__version__)

In [ ]:
## 3b — (Optional) Install official mamba-ssm CUDA backend
# Nếu thành công: train nhanh hơn ~3-5x, ít VRAM hơn ở L=4096.
# Nếu fail (version mismatch): tự fallback về pure-PyTorch — không cần làm gì thêm.
import subprocess, sys, torch

USE_OFFICIAL_MAMBA = True   # đổi False nếu muốn dùng pure-PyTorch

if USE_OFFICIAL_MAMBA and torch.cuda.is_available():
    print("Installing causal-conv1d + mamba-ssm (build ~5-10 phút lần đầu)...")
    r1 = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "causal-conv1d>=1.1.0"], capture_output=True)
    r2 = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mamba-ssm"],            capture_output=True)
    try:
        import mamba_ssm
        print(f"mamba-ssm {mamba_ssm.__version__} OK — official CUDA Mamba sẽ được dùng (3-5x nhanh hơn)")
    except ImportError:
        print("mamba-ssm install failed — sẽ fallback về pure-PyTorch MambaBlock tự động")
        USE_OFFICIAL_MAMBA = False
else:
    print("Skipping mamba-ssm install (USE_OFFICIAL_MAMBA=False hoặc không có GPU)")
    USE_OFFICIAL_MAMBA = False

print(f"USE_OFFICIAL_MAMBA = {USE_OFFICIAL_MAMBA}")

## 4 — Tìm NASA dataset + vào repo

In [ ]:
import os, subprocess
REPO = '/kaggle/working/ai-module'
found = [f for f in subprocess.check_output(['find','/kaggle/input','-name','metadata.csv']).decode().splitlines() if f]
assert found, 'Khong thay metadata.csv — + Add Data dataset NASA cleaned_dataset'
DATASET = os.path.dirname(found[0])
os.chdir(REPO)
print('DATASET    :', DATASET)
print('has data/  :', os.path.isdir(f'{DATASET}/data'))
print('cwd        :', os.getcwd())
print('scaler.pkl :', os.path.isfile('models/weights/scaler.pkl'), '(committed, preprocess_long reuse)')

## 5 — Part A: Standard model v1.4 (window=30, GH-54) — preprocess

Tạo `data/processed/{train,val,test}.pt` (6 features: 4 base + cycle_count + soc_percent) + `models/weights/scaler.pkl` (4-feature) + `feature_scaler.pkl`.

In [ ]:
import os; os.chdir('/kaggle/working/ai-module')
!python scripts/preprocess.py --data-dir "{DATASET}" --output-dir data/processed

## 6 — Part A: Full training (v1.4)

Model nhỏ (D_MODEL=64, 2 layer) — train được cả trên CPU, nhưng GPU nhanh hơn nhiều. `--epochs 100` với early-stop `patience=15` (xem `scripts/train.py`).

In [ ]:
import os; os.chdir('/kaggle/working/ai-module')
!python scripts/train.py --data-dir data/processed --epochs 100 --log-dir logs/training

## 7 — Part A: Kết quả + đóng gói artifact (v1.4)

In [ ]:
import os, glob, shutil, torch, sys
os.chdir('/kaggle/working/ai-module')
sys.path.insert(0, '/kaggle/working/ai-module')
from src.core.config import MAMBA_PATH, ISO_FOREST_PATH, SCALER_PATH, FEATURE_SCALER_PATH

logs = sorted(glob.glob('logs/training/train_*.log'), key=os.path.getmtime)
if logs:
    print('Log:', logs[-1]); print('-'*50)
    !grep -E "Test MAE|Test RMSE|Saved Mamba|Saved IsolationForest" "{logs[-1]}"
if os.path.isfile(MAMBA_PATH):
    c = torch.load(MAMBA_PATH, map_location='cpu', weights_only=False)
    print('-'*50)
    print(f"version={c['version']} window={c['window_size']} input_features={c['input_features']}")
    print(f"Test MAE={c['test_mae']:.4f}%  RMSE={c['test_rmse']:.4f}%")
else:
    print(f'Checkpoint not found: {MAMBA_PATH}')

os.makedirs('/kaggle/working/out_std', exist_ok=True)
for f in [MAMBA_PATH, ISO_FOREST_PATH, SCALER_PATH, FEATURE_SCALER_PATH]:
    if os.path.isfile(f): shutil.copy2(f, '/kaggle/working/out_std/'); print('copied', f)
shutil.make_archive('/kaggle/working/mamba_v1.4_artifacts', 'zip', '/kaggle/working/out_std')
print('\nDownload: Output tab -> mamba_v1.4_artifacts.zip')
print('Commit 4 artifacts (soh_mamba_v1.4.pth, isolation_forest_v1.4.pkl, scaler.pkl, feature_scaler.pkl) vao dev + note MAE/RMSE.')